# 021 — Training: heteroscedastic (Gaussian NLL)

Trains the heteroscedastic counterpart of the four base architectures — `unet_nll`, `resunet_nll`, `attention_unet_nll`, `efficientnet_unet_nll` — each predicting `(mu, log_var)` per pixel instead of a single value, with the plain **Gaussian NLL** loss (`scripts.losses.gaussian_nll_loss`). Same block layout as `020_training.ipynb` — see its title cell for the full table of companion notebooks. `022_training_v2.ipynb` retrains these same four architectures with the beta-weighted NLL variant instead, into a separate checkpoint tree, so neither run overwrites the other.

Only the **artwork-and-mockups** split is used (see §1).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import get_callbacks
from scripts.trainer_nll import compile_model_nll, get_model_nll
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020_training.ipynb` §1 (real artworks grouped and leakage-free; mockup groups split at the pair level) — required so these checkpoints are trained and evaluated under the same conditions as the deterministic ones they are compared against.


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss function — Gaussian NLL

Each architecture outputs two channels per pixel: `mu` (mean, same role as the deterministic models' single output) and `log_var` (log-variance, clipped to `[settings.NLL_LOG_VAR_MIN, settings.NLL_LOG_VAR_MAX]`). The loss (`scripts.losses.gaussian_nll_loss`) is the plain Gaussian negative log-likelihood:

```
loss = 0.5 * exp(-log_var) * (y_true - mu) ** 2 + 0.5 * log_var
```

The first term is an uncertainty-weighted squared error; the second penalises inflating `log_var` to trivially shrink the first. See `code-review.md` §7.6 for the full rationale and references (Nix & Weigend 1994; Kendall & Gal 2017).

Metrics (`mae`, `ssim`, `psnr`) are computed from the `mu` channel only (`scripts.metrics.Mu*Metric`), so they stay directly comparable to the deterministic architectures' metrics in `030_evaluation.ipynb`.


## 3. Train all four NLL architectures

Same loop structure as `020_training.ipynb` §3, using `compile_model_nll` with `loss_name="gaussian_nll"` fixed — this notebook trains only that loss variant; `022_training_v2.ipynb` covers `beta_nll`. Checkpoints go to `models/nll_gaussian/<arch>/best_model.keras`. Set `EPOCHS = 2` for a quick smoke test before committing to a full run.


In [ ]:
ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
LOSS_NAME = "gaussian_nll"
MODEL_DIR = settings.MODELS_DIR / "nll_gaussian"
LOG_DIR = settings.LOGS_DIR / "nll_gaussian"

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}  (loss: {LOSS_NAME})")
    print(f"{'=' * 60}")

    model = get_model_nll(arch)
    model = compile_model_nll(model, lr=settings.LEARNING_RATE, loss_name=LOSS_NAME)
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}, {LOSS_NAME}): {best_val_loss:.4f}")

## 4. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch} ({LOSS_NAME})")
    plt.show()

## 5. Summary

Checkpoints saved to `models/nll_gaussian/<arch>/best_model.keras`. Logs written to `logs/nll_gaussian/<arch>/`.


In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")